# S2.1 · 合成数据生成与正确性验收

本步用结构因果模型（SCM）生成纵向联邦场景的双方数据。**关键在于地面真值已知**：互补性 λ、冗余度、重叠率都是我们设定的，因此可以检验模型学到的增益是否与理论增益一致。

> ⚠️ 合成数据只能验证**机制**，不能替代真实业务数据做效果承诺。

In [1]:
ROUND_DP = 4          # 表格展示精度（不影响任何计算结果）
import sys, subprocess, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "registry").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, yaml
CONFIG_PATH = ROOT / "modules/m2_synthetic/configs/scenarios.yaml"
config = yaml.safe_load(open(CONFIG_PATH, encoding="utf-8"))
seed = config.get("seeds", [config.get("seed")])[0]
git = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True, cwd=ROOT).stdout.strip()
print("config:", CONFIG_PATH.relative_to(ROOT))
print("seed  :", seed, "| 全部种子:", config.get("seeds"))
print("git   :", git or "(未提交)")
print("numpy :", np.__version__, "| pandas:", pd.__version__)

config: modules/m2_synthetic/configs/scenarios.yaml
seed  : 11 | 全部种子: [11, 22, 33, 44, 55]
git   : 6a38f54
numpy : 2.3.5 | pandas: 2.3.3


In [2]:
from modules.m2_synthetic.components.scm_generator import (
    load_scenarios, generate, usable_mask, theoretical_gain_signal)
scenarios = load_scenarios(config)
pd.DataFrame([vars(s) for s in scenarios]).set_index('name')

,n_party_a,overlap_rate,complementarity,redundancy,marginal_drift,size_asymmetry,base_rate,consent_rate,consent_selectivity,match_error_rate,time_drift,label_noise,hte_strength,signal_form,dim_shared,dim_private_a,dim_private_b
name,,,,,,,,,,,,,,,,,
S1_基准,40000,0.3,0.8,0.3,0.0,1.0,0.03,0.6,0.0,0.00,0.0,0.0,0.5,linear,4,6,6
S2_零互补,40000,0.3,0.0,0.3,0.0,1.0,0.03,0.6,0.0,0.00,0.0,0.0,0.5,linear,4,6,6
S3_高互补,40000,0.3,2.0,0.3,0.0,1.0,0.03,0.6,0.0,0.00,0.0,0.0,0.5,linear,4,6,6
S4_高冗余,40000,0.3,0.8,1.2,0.0,1.0,0.03,0.6,0.0,0.00,0.0,0.0,0.5,linear,4,6,6
S5_低重叠高漂移,40000,0.1,0.8,0.3,0.8,1.0,0.03,0.6,0.0,0.00,1.0,0.0,0.5,linear,4,6,6
S6_匹配噪声,40000,0.3,0.8,0.3,0.0,1.0,0.03,0.6,0.0,0.05,0.0,0.0,0.5,linear,4,6,6
S7_同意选择偏差,40000,0.3,0.8,0.3,0.0,1.0,0.03,0.4,1.0,0.00,0.0,0.0,0.5,linear,4,6,6
S8_稀疏正样本,40000,0.3,1.2,0.3,0.0,1.0,0.01,0.6,0.0,0.00,0.0,0.0,0.5,linear,4,6,6


## 各场景的可用样本量与合规折损

`usable = 交集 ∩ 同意`。这一步的折损是**合规成本的直接体现**。

In [3]:
rows = []
for s in scenarios:
    d = generate(s, seed)
    m = usable_mask(d)
    rows.append({'场景': s.name, '总样本': len(d['y']),
                 '交集内': int(d['in_overlap'].sum()),
                 '交集且同意': int(m.sum()),
                 '正样本率': float(d['y_control'][m].mean()),
                 '合规留存率': float(m.mean())})
pd.DataFrame(rows).set_index('场景').round(ROUND_DP)

,总样本,交集内,交集且同意,正样本率,合规留存率
场景,,,,,
S1_基准,40000,11882,7236,0.0258,0.1809
S2_零互补,40000,11882,7236,0.0286,0.1809
S3_高互补,40000,11882,7236,0.0250,0.1809
S4_高冗余,40000,11882,7236,0.0258,0.1809
S5_低重叠高漂移,40000,3993,2469,0.0255,0.0617
S6_匹配噪声,40000,11882,7236,0.0258,0.1809
S7_同意选择偏差,40000,11882,4799,0.0502,0.1200
S8_稀疏正样本,40000,11882,7236,0.0084,0.1809


## 正确性验收：理论增益 vs 实测增益

`theoretical_gain_signal` 用**真实信号**构造两个 oracle 打分（含 B / 不含 B），其 AUC 差就是该场景下 B 侧数据的理论价值上界。

In [4]:
from sklearn.metrics import roc_auc_score
rows = []
for s in scenarios:
    for sd in config['seeds']:
        d = generate(s, sd); m = usable_mask(d); g = theoretical_gain_signal(d)
        y = d['y_control'][m]
        rows.append({'场景': s.name, '种子': sd,
                     '理论增益': roc_auc_score(y, g['with_b'][m])
                                 - roc_auc_score(y, g['without_b'][m])})
th = pd.DataFrame(rows).groupby('场景')['理论增益'].agg(['mean', 'std']).round(ROUND_DP)
th

,mean,std
场景,,
S1_基准,0.0360,0.0050
S2_零互补,0.0000,0.0000
S3_高互补,0.1534,0.0119
S4_高冗余,0.0360,0.0050
S5_低重叠高漂移,0.0355,0.0095
S6_匹配噪声,0.0360,0.0050
S7_同意选择偏差,0.0468,0.0031
S8_稀疏正样本,0.0778,0.0238


**验收判据**：S2_零互补 的理论增益必须为 0——这是生成器实现正确的硬检验。

In [5]:
ZERO_TOL = 1e-9
v = float(th.loc['S2_零互补', 'mean'])
print('S2_零互补 理论增益 =', v)
print('验收:', '通过' if abs(v) < ZERO_TOL else '不通过')

S2_零互补 理论增益 = 0.0
验收: 通过


## S2.3 · 第二种 SCM 结构：检验结论对生成假设的稳健性

上面全部场景的信号都是潜变量的**线性组合**。这带来一个隐患：「调参后 L3a 联邦LR > L3c SplitNN > L3b 纵向GBDT」这个排序，有多少是模型本身的差别，有多少只是**生成方式偏向线性模型**？

分支卡 `BC-M5-001` 的 falsifier 正是为此写下的。这里加两种非线性信号形式：

| 形式 | 信号 | 检验的机制 |
|---|---|---|
| `linear` | s_shared + s_a + λ·s_b | 可加贡献（原场景库） |
| `interaction` | s_shared + s_a + λ·(s_a × s_b) | 被动方**只经交互**起作用 |
| `threshold` | s_shared + s_a + λ·1[s_b > 0] | 被动方以阶跃方式起作用 |

`interaction` 最有理论动机：同样的 s_b，在 s_a 高的人身上是正向、低的人身上是负向。线性模型无论怎么调参都拿不到它（需要显式交叉项），树模型可以逼近。**「双方特征的交互只有联合建模才能捕获」正是纵向联邦的核心论点之一。**

In [6]:
nl_raw = yaml.safe_load(open(
    ROOT / 'modules/m2_synthetic/configs/scenarios_nonlinear.yaml', encoding='utf-8'))
nl = load_scenarios(nl_raw)
rows = []
for c in nl:
    g = [roc_auc_score(generate(c, sd)['y_control'][usable_mask(generate(c, sd))],
                       theoretical_gain_signal(generate(c, sd))['with_b'][usable_mask(generate(c, sd))])
         - roc_auc_score(generate(c, sd)['y_control'][usable_mask(generate(c, sd))],
                         theoretical_gain_signal(generate(c, sd))['without_b'][usable_mask(generate(c, sd))])
         for sd in nl_raw['seeds']]
    rows.append({'场景': c.name, '信号形式': c.signal_form,
                 '互补性': c.complementarity, '理论增益': float(np.mean(g))})
pd.DataFrame(rows).set_index('场景').round(ROUND_DP)

,信号形式,互补性,理论增益
场景,,,
N1_交互_基准,interaction,0.8,0.0378
N2_交互_高,interaction,2.0,0.1767
N3_阶跃_基准,threshold,0.8,0.0325
N4_阶跃_高,threshold,2.0,0.0923
N5_交互_零互补,interaction,0.0,0.0000
N6_线性_对照,linear,0.8,0.0360


**两项验收**：`N5_交互_零互补` 的理论增益必须为 0（λ=0 对三种形式同时成立）；`N6_线性_对照` 与原场景库 `S1_基准` 的理论增益必须一致（两份配置口径相同）。

### 调参后的模型排序：分支卡 falsifier 的判决

In [7]:
nlad = pd.read_csv(ROOT / 'modules/m5_modeling/results/nonlinear_ladder.csv')
COLS = ['N6_线性_对照', 'N1_交互_基准', 'N2_交互_高',
        'N3_阶跃_基准', 'N4_阶跃_高', 'N5_交互_零互补']
piv = nlad.pivot_table(index='level', columns='scenario', values='test_auc',
                       aggfunc='mean')
piv[COLS].round(ROUND_DP)

scenario,N6_线性_对照,N1_交互_基准,N2_交互_高,N3_阶跃_基准,N4_阶跃_高,N5_交互_零互补
level,,,,,,
L0_LR,0.7059,0.7303,0.7506,0.7171,0.6936,0.7160
L1_LR,0.7056,0.7285,0.7483,0.7178,0.6882,0.7148
L2_LR,0.7107,0.7332,0.7484,0.7195,0.6972,0.7158
L3a_联邦LR,0.7814,0.7512,0.7903,0.7810,0.8066,0.7399
L3b_纵向GBDT,0.7416,0.7405,0.8193,0.7534,0.7926,0.7201
L3c_SplitNN,0.7651,0.7465,0.7966,0.7626,0.7843,0.7077
L4_LR,0.7815,0.7507,0.7901,0.7815,0.8071,0.7407


In [8]:
L3 = ['L3a_联邦LR', 'L3b_纵向GBDT', 'L3c_SplitNN']
for sc in COLS:
    order = piv.loc[L3, sc].sort_values(ascending=False)
    print(f"{sc:14s} " + ' > '.join(f'{k}({v:.4f})' for k, v in order.items()))

N6_线性_对照       L3a_联邦LR(0.7814) > L3c_SplitNN(0.7651) > L3b_纵向GBDT(0.7416)
N1_交互_基准       L3a_联邦LR(0.7512) > L3c_SplitNN(0.7465) > L3b_纵向GBDT(0.7405)
N2_交互_高        L3b_纵向GBDT(0.8193) > L3c_SplitNN(0.7966) > L3a_联邦LR(0.7903)
N3_阶跃_基准       L3a_联邦LR(0.7810) > L3c_SplitNN(0.7626) > L3b_纵向GBDT(0.7534)
N4_阶跃_高        L3a_联邦LR(0.8066) > L3b_纵向GBDT(0.7926) > L3c_SplitNN(0.7843)
N5_交互_零互补      L3a_联邦LR(0.7399) > L3b_纵向GBDT(0.7201) > L3c_SplitNN(0.7077)


In [9]:
PCT = 100
print('C1 稳健性：L1 捕获比例')
for sc in COLS:
    l0, l1 = piv.loc['L0_LR', sc], piv.loc['L1_LR', sc]
    best3 = piv.loc[L3, sc].max()
    share = (l1 - l0) / (best3 - l0) * PCT if best3 > l0 else float('nan')
    print(f'  {sc:14s} L1−L0={l1-l0:+.4f}  最优L3−L0={best3-l0:+.4f}  L1 捕获 {share:5.1f}%')

C1 稳健性：L1 捕获比例
  N6_线性_对照       L1−L0=-0.0003  最优L3−L0=+0.0755  L1 捕获  -0.4%
  N1_交互_基准       L1−L0=-0.0018  最优L3−L0=+0.0209  L1 捕获  -8.7%
  N2_交互_高        L1−L0=-0.0023  最优L3−L0=+0.0687  L1 捕获  -3.4%
  N3_阶跃_基准       L1−L0=+0.0007  最优L3−L0=+0.0638  L1 捕获   1.0%
  N4_阶跃_高        L1−L0=-0.0053  最优L3−L0=+0.1130  L1 捕获  -4.7%
  N5_交互_零互补      L1−L0=-0.0012  最优L3−L0=+0.0239  L1 捕获  -5.0%
